# RAQA — One-Run Quickstart Demo

Runs the **entire RAQA process end-to-end in one notebook**, with **no external servers and no API
key**: synthesize a tiny sample → build a lightweight **SQLite** database → pose a couple of
*slightly complex* benchmark questions (with gold answers) → let the **learned RAQA** (the real
pretrained `r_θ` risk + `c_θ` cost predictors) choose a query plan, execute it, and score it
(execution success / accuracy / Run-but-Wrong).

> ⚠️ **Synthetic & MIMIC-IV-only.** Real MIMIC-IV is credentialed (PhysioNet) and is **not** included;
> the AMC cohort used in the paper cannot be exported. Data here is **random** (correct schema,
> meaningless values) — a smoke test of the mechanics, not a clinical result.

**Sections.** (1) synthetic data → (2) SQLite DB → (3) mini benchmark → (4) RAQA select + execute +
judge → (5) *optional* full pipeline on real MIMIC with MySQL/Neo4j + OpenAI.

In [1]:
import os, sys, re, sqlite3, random, warnings, pathlib
import numpy as np, pandas as pd, joblib
warnings.filterwarnings("ignore")

ROOT = pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
os.environ.setdefault("DATA_DIR", str(ROOT / "demo_data"))
os.environ.setdefault("ARTIFACT_DIR", str(ROOT / "demo_artifacts"))
import config

# the real pretrained learned RAQA components used in the paper
R_THETA = joblib.load(os.path.join(config.MODEL_DIR, "r_theta_lightgbm_isotonic.joblib"))
C_THETA = joblib.load(os.path.join(config.MODEL_DIR, "c_theta_lightgbm.joblib"))
print("loaded r_theta, c_theta from", os.path.relpath(config.MODEL_DIR, ROOT))

loaded r_theta, c_theta from models


## 1. Synthetic sample (unified schema)

A handful of patients with admissions, diagnoses, labs, and medications (random values, correct
schema).

In [2]:
rng = random.Random(20260607)
LABS  = ["Creatinine","Hemoglobin","Potassium","Glucose"]
MEDS  = ["Heparin","Insulin","Acetaminophen","Furosemide","Vancomycin"]
DIAGS = ["Hypertension","Acute kidney failure","Pneumonia","Type 2 diabetes"]

patient, admission, diagnosis, lab, medi = [], [], [], [], []
for i in range(1, 9):
    pid = f"10{i:02d}"
    patient.append((pid, rng.choice(["M","F"]), rng.randint(40,85)))
    for h in range(1, rng.randint(1,2)+1):
        aid = f"{pid}{h}"
        admission.append((pid, aid, f"{pid}_{h}"))
        for s, d in enumerate(rng.sample(DIAGS, rng.randint(2,4)), 1):
            diagnosis.append((pid, aid, d, s))
        for _ in range(rng.randint(3,8)):
            lab.append((pid, aid, rng.choice(LABS), round(rng.uniform(0.5, 180), 1)))
        for _ in range(rng.randint(2,6)):
            medi.append((pid, aid, rng.choice(MEDS)))

T = {
 "Patient":   pd.DataFrame(patient,   columns=["patientId","gender","anchor_age"]),
 "Admission": pd.DataFrame(admission, columns=["patientId","admissionId","admissionKey"]),
 "Diagnosis": pd.DataFrame(diagnosis, columns=["patientId","admissionId","diagnosisName","seq_num"]),
 "Lab":       pd.DataFrame(lab,       columns=["patientId","admissionId","testName","resultNumericValue"]),
 "Medi":      pd.DataFrame(medi,      columns=["patientId","admissionId","genericName"]),
}
for n, df in T.items():
    print(f"  {n:10s} {len(df):4d} rows")
T["Lab"].head()

  Patient       8 rows
  Admission    10 rows
  Diagnosis    29 rows
  Lab          63 rows
  Medi         40 rows


,patientId,admissionId,testName,resultNumericValue
0,1001,10011,Glucose,53.2
1,1001,10011,Hemoglobin,33.7
2,1001,10011,Glucose,166.8
3,1001,10011,Glucose,173.4
4,1001,10011,Glucose,112.9


## 2. Build a lightweight DB (SQLite — no server needed)

The paper uses MySQL (RDB) + Neo4j (GDB); here we load the same unified tables into in-process
**SQLite** so the demo needs no servers.

In [3]:
con = sqlite3.connect(":memory:")
for name, df in T.items():
    df.to_sql(name, con, index=False, if_exists="replace")

def run_sql(q):
    # execute SQL on the demo DB; returns (ok, rows_or_error)
    try:
        return True, con.execute(q).fetchall()
    except Exception as e:
        return False, str(e)

print("tables:", [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")])
print("sanity:", run_sql("SELECT COUNT(*) FROM Lab"))

tables: ['Patient', 'Admission', 'Diagnosis', 'Lab', 'Medi']
sanity: (True, [(63,)])


## 3. Mini benchmark — two *slightly complex* questions

A **multi-hop** question (diagnosis → labs in the same admission) and a **cross-source aggregate**.
Each pairs a correct plan (`A`) with a riskier **over-joined** alternative (`B`) that still executes
but fan-outs into an **inflated / wrong** answer — the Run-but-Wrong trap RAQA should avoid.

In [4]:
benchmark = [
  {"qid":1, "difficulty":"complex",
   "ctx":{"category":"patient_centric_C4_multi_hop_association","difficulty":"complex","query_type":"RDB"},
   "question":"For patient 1001, how many lab tests were recorded during admissions in which the "
              "patient was diagnosed with acute kidney failure?",
   "gold_sql":"SELECT COUNT(*) FROM Lab WHERE patientId='1001' AND admissionId IN "
              "(SELECT admissionId FROM Diagnosis WHERE patientId='1001' AND diagnosisName='Acute kidney failure')",
   "candidates":[
     {"plan_id":"A","db_type":"RDB","predicted_error_risk":0.25,
      "query":"SELECT COUNT(*) FROM Lab WHERE patientId='1001' AND admissionId IN "
              "(SELECT admissionId FROM Diagnosis WHERE patientId='1001' AND diagnosisName='Acute kidney failure')"},
     {"plan_id":"B","db_type":"RDB","predicted_error_risk":0.55,   # fan-out join -> inflated, wrong
      "query":"SELECT COUNT(*) FROM Lab l JOIN Medi m ON l.patientId=m.patientId WHERE l.patientId='1001'"}]},
  {"qid":2, "difficulty":"complex",
   "ctx":{"category":"population_centric_C5_hybrid_cross_source","difficulty":"complex","query_type":"RDB"},
   "question":"Across all patients, what is the average Creatinine result value?",
   "gold_sql":"SELECT ROUND(AVG(resultNumericValue),2) FROM Lab WHERE testName='Creatinine'",
   "candidates":[
     {"plan_id":"A","db_type":"RDB","predicted_error_risk":0.25,
      "query":"SELECT ROUND(AVG(resultNumericValue),2) FROM Lab WHERE testName='Creatinine'"},
     {"plan_id":"B","db_type":"RDB","predicted_error_risk":0.55,   # medi-weighted avg -> wrong
      "query":"SELECT ROUND(AVG(l.resultNumericValue),2) FROM Lab l JOIN Medi m "
              "ON l.patientId=m.patientId WHERE l.testName='Creatinine'"}]},
]
for b in benchmark:
    _, g = run_sql(b["gold_sql"]); b["gold"] = g
    print(f"  Q{b['qid']} [{b['difficulty']}] gold={g[0][0]}: {b['question'][:72]}...")

  Q1 [complex] gold=6: For patient 1001, how many lab tests were recorded during admissions in ...
  Q2 [complex] gold=94.69: Across all patients, what is the average Creatinine result value?...


## 4. RAQA in action — learned plan selection, execution, judging

For each candidate plan we build the feature vector, predict **risk** with `r_θ` and **cost
(latency)** with `c_θ`, and pick the plan minimizing $J = \hat{C}_\theta + \lambda\, r_\theta + \pi$
($\pi=0$, $\lambda=1$). We execute the selected plan, compare to gold, and also show what the
*rejected* plan would have returned (the avoided Run-but-Wrong).

In [5]:
QFK = ["q_len","has_optional","has_distinct","has_count","has_match","has_join","has_group_by",
       "has_order_by","has_with","is_multi_step","step_count","optional_count","q_has_next"]
def qfeats(q):
    s = q.lower(); sc = max(len(re.findall(r"step\s*\d+", s)), 1)
    return {"q_len":min(len(q),5000),"has_optional":int("optional" in s),"has_distinct":int("distinct" in s),
        "has_count":int("count(" in s),"has_match":int("match(" in s),"has_join":int(" join " in s),
        "has_group_by":int("group by" in s),"has_order_by":int("order by" in s),
        "has_with":int(" with " in s or s.startswith("with ")),"is_multi_step":int(sc>=2),
        "step_count":sc,"optional_count":s.count("optional"),"q_has_next":int(":next" in s)}
def feat(plan, ctx):
    f = qfeats(plan["query"]); f["db_type"] = plan["db_type"]; f.update(ctx)
    f["planner_risk"] = plan["predicted_error_risk"]; return f

def raqa_select(cands, ctx, lam=1.0, verbose=False):
    X = pd.DataFrame([feat(p, ctx) for p in cands])
    risk = R_THETA.predict_proba(X)[:, 1]              # r_theta
    cost = np.exp(C_THETA.predict(X))                  # c_theta -> latency (s)
    lo, hi = cost.min(), cost.max(); chat = (cost - lo) / (hi - lo + 1e-9)
    for p, r, c, ch in zip(cands, risk, cost, chat):
        p["J"] = float(ch) + lam * float(r)
        if verbose:
            print(f"      plan {p['plan_id']} ({p['db_type']:3s})  r_theta={r:.3f}  c_theta={c:5.1f}s  -> J={p['J']:.3f}")
    return min(cands, key=lambda p: p["J"])

val = lambda res: res[0][0] if (isinstance(res, list) and res) else res

rows = []
for b in benchmark:
    print(f"Q{b['qid']}: {b['question']}")
    sel = raqa_select(b["candidates"], b["ctx"], verbose=True)
    ok, res = run_sql(sel["query"])
    exec_ok = int(bool(ok and res)); correct = int(exec_ok and res == b["gold"]); rbw = int(exec_ok and not correct)
    rej = [c for c in b["candidates"] if c is not sel][0]
    ok2, res2 = run_sql(rej["query"]); rej_wrong = bool(ok2 and res2 and res2 != b["gold"])
    print(f"   -> RAQA selects {sel['plan_id']}: exec={exec_ok} correct={correct} RbW={rbw}  "
          f"(got {val(res)}, gold {val(b['gold'])})")
    print(f"      rejected {rej['plan_id']} would return {val(res2)}  ->  "
          f"{'WRONG (Run-but-Wrong avoided)' if rej_wrong else 'also ok'}\n")
    rows.append({"qid":b["qid"],"selected":sel["plan_id"],"exec":exec_ok,"correct":correct,"rbw":rbw})

summary = pd.DataFrame(rows)
print("=== summary ==="); print(summary.to_string(index=False))
print(f"\nExecution success: {summary['exec'].mean()*100:.0f}%  |  "
      f"Accuracy: {summary['correct'].mean()*100:.0f}%  |  RbW: {summary['rbw'].mean()*100:.0f}%")

Q1: For patient 1001, how many lab tests were recorded during admissions in which the patient was diagnosed with acute kidney failure?
      plan A (RDB)  r_theta=0.102  c_theta=  7.8s  -> J=0.102
      plan B (RDB)  r_theta=0.051  c_theta=  8.1s  -> J=1.051
   -> RAQA selects A: exec=1 correct=1 RbW=0  (got 6, gold 6)
      rejected B would return 12  ->  WRONG (Run-but-Wrong avoided)

Q2: Across all patients, what is the average Creatinine result value?
      plan A (RDB)  r_theta=0.051  c_theta=  7.5s  -> J=0.051
      plan B (RDB)  r_theta=0.051  c_theta=  9.1s  -> J=1.051
   -> RAQA selects A: exec=1 correct=1 RbW=0  (got 94.69, gold 94.69)
      rejected B would return 93.75  ->  WRONG (Run-but-Wrong avoided)

=== summary ===
 qid selected  exec  correct  rbw
   1        A     1        1    0
   2        A     1        1    0

Execution success: 100%  |  Accuracy: 100%  |  RbW: 0%


## 5. *(Optional)* Full pipeline on real MIMIC-IV

The full pipeline from the paper uses MySQL + Neo4j + an OpenAI backbone, via the bundled scripts.
It runs only if the environment is configured; otherwise the commands are printed.

In [6]:
import subprocess
ready = all(os.environ.get(k) for k in ["MIMIC_MYSQL_URI","MIMIC_NEO4J_URI","OPENAI_API_KEY"])
if ready:
    for cmd in (["db/build_mimic_db.py"], ["generation/gen_benchmark.py"], ["experiment/run_methods.py","mimic"]):
        print("$ python", *cmd)
        print(subprocess.run([sys.executable, *cmd], cwd=str(ROOT), capture_output=True, text=True).stdout[-1500:])
else:
    print("[skipped] full pipeline needs servers + key. To run it:\n")
    print("  export MIMIC_MYSQL_URI=...  MIMIC_NEO4J_URI=...  NEO4J_USER=...  NEO4J_PASSWORD=...  OPENAI_API_KEY=...")
    print("  python db/build_mimic_db.py")
    print("  python generation/gen_benchmark.py        # V7_TARGET=2 for a smoke run")
    print("  python experiment/run_methods.py mimic")

[skipped] full pipeline needs servers + key. To run it:

  export MIMIC_MYSQL_URI=...  MIMIC_NEO4J_URI=...  NEO4J_USER=...  NEO4J_PASSWORD=...  OPENAI_API_KEY=...
  python db/build_mimic_db.py
  python generation/gen_benchmark.py        # V7_TARGET=2 for a smoke run
  python experiment/run_methods.py mimic


## Summary

Sections 1–4 ran the **whole RAQA mechanism end-to-end on synthetic data, zero external deps** —
build DB → benchmark → learned plan selection (`r_θ`/`c_θ`) → execute → judge — and showed RAQA
choosing the correct plan while the over-joined alternative would have produced a Run-but-Wrong
error. For a faithful reproduction, supply real MIMIC-IV + servers/key (Section 5); the pretrained
learned components are the same ones used in the paper.